# 05C — DBSCAN : segmentation par densité | Panier Vert

Chaque notebook lit **le même** `data/gold/user_segments.parquet` produit par le notebook 04. Les transformations (`log1p`, normalisation) et l'échantillon pseudo-aléatoire de 15 000 clients sont identiques : la comparaison est cohérente. Les résultats sont tracés dans `mlruns/` à la racine du projet ; aucune instance MLflow ne doit être lancée pour exécuter les notebooks. Pour afficher l'interface, depuis la racine : `mlflow ui --backend-store-uri ./mlruns`.

**Note méthodologique :** la normalisation est ajustée ici sur l'ensemble des clients avant l'échantillonnage, comme dans le notebook initial. Pour une évaluation hors échantillon stricte, il faudrait ajuster les transformations sur un jeu d'entraînement seulement. Les métriques internes de clustering ne garantissent pas à elles seules une segmentation utile métier.

**Attention :** la silhouette DBSCAN porte sur les points classés seulement ; elle doit être lue avec le pourcentage de bruit.

## 1 — Chargement et préparation

In [ ]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

cwd = Path.cwd()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / "data" / "gold" / "user_segments.parquet").exists()), None)
if ROOT is None:
    raise FileNotFoundError("data/gold/user_segments.parquet introuvable : exécuter le notebook 04 ou adapter ROOT.")
GOLD = ROOT / "data" / "gold" / "user_segments.parquet"
REPORT_DIR = ROOT / "data" / "reports" / "segmentation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri((ROOT / "mlruns").resolve().as_uri())

df = pd.read_parquet(GOLD).set_index("user_id")
print(f"{len(df):,} clients | {df.shape[1]} colonnes")

# Préparation identique pour les trois modèles
DROP = ["tenure_days", "nb_distinct_products", "product_diversity", "segment", "segment_name"]
feat = df.drop(columns=[c for c in DROP if c in df.columns]).select_dtypes(include=["number"]).copy()
if feat.empty:
    raise ValueError("Aucune variable numérique disponible après exclusion.")
if feat.isna().any().any():
    raise ValueError("Valeurs manquantes dans les variables : vérifier le notebook 04.")

X = feat.copy()
skewed = [c for c in X.columns if X[c].skew() > 1 and (X[c] >= 0).all()]
X[skewed] = np.log1p(X[skewed])
scaler = StandardScaler()
Z = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)
K = int(df["segment"].nunique()) if "segment" in df.columns else 4
CMP_N = 15_000
rng = np.random.RandomState(RANDOM_STATE)
cmp_idx = rng.choice(len(Z), size=min(CMP_N, len(Z)), replace=False)
Zc = Z.iloc[cmp_idx].to_numpy()
print("Variables :", list(Z.columns))
print("Échantillon commun :", Zc.shape, "| K de référence :", K)

def segmentation_metrics(data, labels, *, exclude_noise=True):
    labels = np.asarray(labels)
    mask = (labels != -1) if exclude_noise else np.ones(len(labels), dtype=bool)
    unique = np.unique(labels[mask])
    out = {"n_clusters": int(len(unique)), "noise_pct": float(100 * (~mask).mean()),
           "classified_pct": float(100 * mask.mean())}
    if len(unique) >= 2 and mask.sum() > len(unique):
        # Silhouette sur un sous-échantillon pour éviter la matrice O(n²)
        out["silhouette"] = float(silhouette_score(data[mask], labels[mask],
            sample_size=min(2500, int(mask.sum())), random_state=RANDOM_STATE))
        out["davies_bouldin"] = float(davies_bouldin_score(data[mask], labels[mask]))
    else:
        out["silhouette"] = float("nan")
        out["davies_bouldin"] = float("nan")
    if mask.any():
        out["largest_cluster_pct"] = float(100 * pd.Series(labels[mask]).value_counts(normalize=True).iloc[0])
    return out

def log_numeric_metrics(d):
    for k, v in d.items():
        if isinstance(v, (int, float, np.integer, np.floating)) and np.isfinite(v):
            mlflow.log_metric(k, float(v))


## 2 — Graphe des distances au voisin le plus proche

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
MIN_SAMPLES = max(2, 2 * Zc.shape[1])
nn = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(Zc)
kdist = np.sort(nn.kneighbors(Zc)[0][:, -1])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(kdist)
ax.set(xlabel="Clients triés", ylabel=f"Distance au {MIN_SAMPLES}e voisin",
       title="DBSCAN — graphe des k-distances")
fig.tight_layout(); fig.savefig(REPORT_DIR / "dbscan_kdistance.png", dpi=130)
plt.show()
display(pd.Series({p: np.percentile(kdist, p) for p in [50, 60, 70, 80, 90, 95]}).to_frame("eps"))

## 3 — Grille de valeurs eps et MLflow

DBSCAN peut être coûteux sur 15 000 clients. Chaque configuration est sauvegardée dans MLflow.

In [ ]:
mlflow.set_experiment("PanierVert_DBSCAN_Clients")
rows = []
for percentile in [50, 60, 70, 80, 90, 95]:
    eps = float(np.percentile(kdist, percentile))
    with mlflow.start_run(run_name=f"dbscan_p{percentile}"):
        t0 = time.perf_counter()
        labels = DBSCAN(eps=eps, min_samples=MIN_SAMPLES, n_jobs=-1).fit_predict(Zc)
        duration = time.perf_counter() - t0
        scores = segmentation_metrics(Zc, labels)
        scores["duration_s"] = duration
        mlflow.log_params({"model": "DBSCAN", "eps": eps, "eps_percentile": percentile,
                           "min_samples": MIN_SAMPLES, "sample_size": len(Zc)})
        log_numeric_metrics(scores)
        rows.append({"percentile": percentile, "eps": eps, **scores})
        print(f"p{percentile}: {scores['n_clusters']} clusters, {scores['noise_pct']:.1f}% bruit")
db_grid = pd.DataFrame(rows).set_index("percentile")
display(db_grid)
db_grid.to_csv(REPORT_DIR / "dbscan_grille.csv")

## 4 — Sélection et visualisation

Une solution avec un seul cluster ou presque tous les points en bruit n’est pas une segmentation exploitable. Si aucune configuration n’est satisfaisante, on le signale sans prétendre le contraire.

In [ ]:
EPS_MANUEL = None
ok = db_grid[(db_grid.n_clusters.between(2, 10)) &
             (db_grid.noise_pct <= 30) &
             (db_grid.largest_cluster_pct < 90) &
             db_grid.silhouette.notna()]
if EPS_MANUEL is not None:
    EPS = float(EPS_MANUEL)
    justification = "valeur choisie manuellement"
elif not ok.empty:
    best_idx = ok.silhouette.idxmax()
    EPS = float(ok.loc[best_idx, "eps"])
    justification = "meilleure silhouette parmi les solutions filtrées"
else:
    EPS = float(np.percentile(kdist, 90))
    justification = "repli technique (aucune configuration acceptable)"
print(f"EPS retenu : {EPS:.3f} — {justification}")
with mlflow.start_run(run_name="dbscan_selection_finale"):
    labels = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, n_jobs=-1).fit_predict(Zc)
    metrics = segmentation_metrics(Zc, labels)
    mlflow.log_params({"model": "DBSCAN", "eps": EPS, "min_samples": MIN_SAMPLES,
                       "sample_size": len(Zc), "selection": justification})
    log_numeric_metrics(metrics)
    display(pd.DataFrame([metrics]))
    xy = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Zc)
    fig, ax = plt.subplots(figsize=(7, 5))
    noise = labels == -1
    ax.scatter(xy[noise, 0], xy[noise, 1], c="lightgrey", s=5, label="Bruit")
    ax.scatter(xy[~noise, 0], xy[~noise, 1], c=labels[~noise], cmap="tab10", s=5, alpha=.65)
    if noise.any(): ax.legend()
    ax.set(title="DBSCAN — projection PCA", xlabel="PC1", ylabel="PC2")
    fig.tight_layout(); fig.savefig(REPORT_DIR / "dbscan_pca.png", dpi=130)
    mlflow.log_artifact(str(REPORT_DIR / "dbscan_pca.png"))
    mlflow.log_artifact(str(REPORT_DIR / "dbscan_grille.csv"))
    plt.show()